# Researcher name matching
### A walkthrough using the synthetic pipeline configuration

Follow the steps below to load the configured synthetic records, run the name-matching stage, and inspect candidate matches. All people and publication records are fictional. The disambiguation example uses the real TU Wien affiliation mapping solely to demonstrate current-affiliation selection.

## What this notebook does

It loads `config/pipeline_synthetic.toml` and invokes the configured name-matching workflow. The workflow validates its input and output schemas, applies existing Scopus-ID passthrough, and writes `resolved_researchers.csv` to the dedicated `data/output/synthetic/name_matching/` directory. Rerunning the matching cell replaces that synthetic artifact; no research output is touched.

The synthetic organizations are not in the production affiliation map, so the config disables organization filtering. Name matching can return multiple candidate profiles for one researcher; a match is evidence to review, not an automatic one-to-one identity decision.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Find the repository root so paths work when Jupyter starts in a subdirectory.
repo_root = Path.cwd()
while not (repo_root / 'pyproject.toml').is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / 'pyproject.toml').is_file():
    raise RuntimeError('Start Jupyter from within the academic-inbreeding repository.')
sys.path.insert(0, str(repo_root / 'src'))

from acib.constants import (
    AFFILIATION_NAME,
    AUTHOR_ID,
    EID,
    RESEARCHER_ID,
    UNIVERSITY_NAME,
)
from acib.name_matching.config import load_name_matching_config
from acib.name_matching.constants import (
    AUTHOR_NAME,
    MATCH_PRIORITY,
    NAME,
    PERSON_FULLNAME,
)
from acib.name_matching.workflow import run_name_matching

## 1. Load the synthetic pipeline configuration

In [2]:
config_path = repo_root / 'config' / 'pipeline_synthetic.toml'
config = load_name_matching_config(config_path)
config.check_paths_exist()

print(f'Config: {config_path.relative_to(repo_root)}')
print(f'Researcher input: {config.researcher_file_path.relative_to(repo_root)}')
print(f'Candidate-author input: {config.candidate_authors_file.relative_to(repo_root)}')
print(f'Organization filtering: {config.enable_organisation_filter}')
print(f'Existing Scopus ID column: {config.existing_scopus_author_id_column}')
print(f'Output artifact: {config.outputs.resolved_researchers.relative_to(repo_root)}')

Config: config/pipeline_synthetic.toml
Researcher input: data/test/synthetic/researchers.csv
Candidate-author input: data/test/synthetic/scopus_authors.csv
Organization filtering: False
Existing Scopus ID column: scopus_author_id
Output artifact: data/output/synthetic/name_matching/resolved_researchers.csv


## 2. Preview the synthetic input records

Use the project’s schema field constants for DataFrame column access, rather than repeating raw column-name strings.

In [3]:
researchers = pd.read_csv(config.researcher_file_path, encoding=config.encoding)
researchers[[RESEARCHER_ID, PERSON_FULLNAME, UNIVERSITY_NAME]]

,sc_id,person_fullname,university_name
0,990001,Max Mustermann,Example Technical Institute
1,990002,John Doe,Example Technical Institute
2,990003,Riley Unmatched,Example Technical Institute
3,990004,Ada Lovelace,Example Technical Institute
4,990005,Noor Al-Hassan,Example Medical Institute
5,990006,Zoë García,Example Arts College
6,990007,Jean-Luc Martin,Example Technical Institute
7,990008,Alice Marie Nguyen,Example Medical Institute
8,990009,Sami O'Connell,Example Arts College
9,990010,Ana María de la Cruz,Example Social Science Institute


In [4]:
authors = pd.read_csv(config.candidate_authors_file, encoding=config.encoding)
authors[[AUTHOR_ID, EID, NAME, AFFILIATION_NAME]]

,author_id,eid,name,affiliation_name
0,990000001,SYN-PUB-001,"Mustermann, Max",Example Technical Institute
1,990000002,SYN-PUB-002,"Mustermann, M.",Example Technical Institute
2,990000003,SYN-PUB-003,"Mustermann, Max",Example Medical Institute
3,990000004,SYN-PUB-004,"Doe, John",Example Technical Institute
4,990000005,SYN-PUB-005,"Lovelace, Ada",Example Technical Institute
5,990000006,SYN-PUB-006,"Al-Hassan, Noor",Example Medical Institute
6,990000007,SYN-PUB-007,"García, Zoë",Example Arts College
7,990000008,SYN-PUB-008,"Martin, Jean-Luc",Example Technical Institute
8,990000009,SYN-PUB-009,"Nguyen, Alice M.",Example Medical Institute
9,990000010,SYN-PUB-010,"O'Connell, Sami",Example Arts College


## 3. Run name matching

This calls the stage workflow configured above. It validates the CSV inputs, applies the configured existing-ID behavior and matching rules, restores researcher metadata, validates the result, and writes the canonical stage artifact.

In [5]:
resolved = run_name_matching(config, overwrite=True)
print(f'Wrote {len(resolved):,} resolved rows to {config.outputs.resolved_researchers}')

2026-11-03 09:34:22 [info     ] Loading name matching inputs   candidate_authors_file=/home/chri/PycharmProjects/academic-inbreeding/data/test/synthetic/scopus_authors.csv researcher_file_path=/home/chri/PycharmProjects/academic-inbreeding/data/test/synthetic/researchers.csv
2026-11-03 09:34:22 [info     ] Loaded name matching inputs    candidate_author_rows=21 researcher_rows=19
2026-11-03 09:34:22 [info     ] Skipping researchers with existing Scopus author IDs existing_scopus_author_id_column=scopus_author_id skipped_rows=1
2026-11-03 09:34:22 [info     ] Preparing researcher and candidate author names candidate_author_rows=21 researcher_rows=18
2026-11-03 09:34:22 [info     ] Finished matching              matched_researchers=20 total_researchers=22
2026-11-03 09:34:22 [info     ] Writing resolved name matches  output_path=/home/chri/PycharmProjects/academic-inbreeding/data/output/synthetic/name_matching/resolved_researchers.csv output_rows=21
2026-11-03 09:34:22 [info     ] Finish

## 4. Review the resolved candidates

Lower positive priority numbers indicate stronger matching evidence. Priority 0 means an existing Scopus author ID was supplied in the researcher input and passed through without name matching.

In [6]:
result_columns = [RESEARCHER_ID, PERSON_FULLNAME, AUTHOR_ID, EID, AUTHOR_NAME, MATCH_PRIORITY]
matches = resolved[result_columns].sort_values(
    [RESEARCHER_ID, MATCH_PRIORITY, AUTHOR_ID], kind='mergesort'
)
matches

,sc_id,person_fullname,author_id,eid,author_name,priority
0,990001,Max Mustermann,990000001,SYN-PUB-001,"mustermann, max",1
1,990001,Max Mustermann,990000003,SYN-PUB-003,"mustermann, max",1
2,990001,Max Mustermann,990000002,SYN-PUB-002,"mustermann, m",4
3,990002,John Doe,990000004,SYN-PUB-004,"do, john",1
4,990004,Ada Lovelace,990000005,SYN-PUB-005,"lovelace, ada",1
5,990005,Noor Al-Hassan,990000006,SYN-PUB-006,"al hassan, noor",1
6,990006,Zoë García,990000007,SYN-PUB-007,"garcia, zo",1
7,990007,Jean-Luc Martin,990000008,SYN-PUB-008,"martin, jean luc",1
8,990008,Alice Marie Nguyen,990000009,SYN-PUB-009,"nguyen, alice m",3
9,990009,Sami O'Connell,990000010,SYN-PUB-010,"oconnell, sami",1


Multiple rows for one researcher are intentional: they show alternative candidate profiles for human review. The following summary makes those candidate counts easy to scan.

In [7]:
candidate_counts = (
    matches.loc[matches[MATCH_PRIORITY].gt(0)]
    .groupby([RESEARCHER_ID, PERSON_FULLNAME], as_index=False)
    .size()
    .rename(columns={'size': 'candidate_count'})
)
candidate_counts.sort_values('candidate_count', ascending=False)

,sc_id,person_fullname,candidate_count
0,990001,Max Mustermann,3
9,990011,Casey Morgan,2
15,990019,Ari Example,2
1,990002,John Doe,1
4,990006,Zoë García,1
5,990007,Jean-Luc Martin,1
2,990004,Ada Lovelace,1
3,990005,Noor Al-Hassan,1
7,990009,Sami O'Connell,1
6,990008,Alice Marie Nguyen,1


## 5. Check the designed examples

The fixture includes same-name profiles across affiliations, an existing-ID passthrough case, and two researchers with no candidate profiles. These checks confirm those examples are represented in the current output.

In [8]:
mustermann_eids = set(matches.loc[matches[RESEARCHER_ID].eq(990001), EID].dropna())
casey_eids = set(matches.loc[matches[RESEARCHER_ID].eq(990011), EID].dropna())
assert {'SYN-PUB-001', 'SYN-PUB-003'}.issubset(mustermann_eids)
assert {'SYN-PUB-012', 'SYN-PUB-019'}.issubset(casey_eids)
assert matches.loc[matches[RESEARCHER_ID].eq(990013), MATCH_PRIORITY].eq(0).all()
unmatched_ids = set(researchers[RESEARCHER_ID]) - set(resolved[RESEARCHER_ID])
assert {990003, 990012}.issubset(unmatched_ids)
print('All designed candidate, ambiguity, passthrough, and unmatched cases are present.')

All designed candidate, ambiguity, passthrough, and unmatched cases are present.
